In [ ]:
#Copy a LL from CPU to GPU

%%cuda --compiler-args "-rdc=true -lcudadevrt"
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

typedef struct Node {
    char* roll;
    char* name;
    char* ad;
    struct Node* next;
} Node;

__global__ void gpu_ll(Node* head) {
    Node* temp = head;
    printf("%-15s %-20s %-15s\n", "Roll Number", "Name", "Faculty Advisor");
    printf("--------------------------------------------------------\n");
    while (temp != NULL) {
        printf("%-15s %-20s %-15s\n", temp->roll, temp->name, temp->ad);
        temp = temp->next;
    }
}

Node* createNode(const char* roll, const char* student_name, const char* advisor) {
    Node* newNode = (Node*)malloc(sizeof(Node));
    if (newNode == NULL) {
        printf("Memory allocation failed for Node!\n");
        exit(1);
    }

    newNode->roll = strdup(roll);
    newNode->name        = strdup(student_name);
    newNode->ad     = strdup(advisor);
    newNode->next        = NULL;

    return newNode;
}

void insertAtTail(Node** headRef, const char* roll, const char* student_name, const char* advisor) {
    Node* newNode = createNode(roll, student_name, advisor);

    if (*headRef == NULL) {
        *headRef = newNode;
        return;
    }

    Node* temp = *headRef;
    while (temp->next != NULL) {
        temp = temp->next;
    }
    temp->next = newNode;
}

void printList(Node* head) {
    Node* temp = head;
    printf("%-15s %-20s %-15s\n", "Roll Number", "Name", "Faculty Advisor");
    printf("--------------------------------------------------------\n");

    while (temp != NULL) {
        printf("%-15s %-20s %-15s\n", temp->roll, temp->name, temp->ad);
        temp = temp->next;
    }
}

void freeList(Node** headRef) {
    Node* current = *headRef;
    Node* nextNode;

    while (current != NULL) {
        nextNode = current->next;

        free(current->roll);
        free(current->name);
        free(current->ad);

        free(current);
        current = nextNode;
    }
    *headRef = NULL;
}

int main() {
    Node* head = NULL;
    insertAtTail(&head, "24CS001", "Aarav Sharma", "Dr. Kapoor");
    insertAtTail(&head, "24CS002", "Diya Iyer",    "Dr. Menon");
    insertAtTail(&head, "24CS003", "Kabir Singh",  "Prof. Das");
    insertAtTail(&head, "24CS004", "Arindom Bora",  "Prof. Khapra");
    insertAtTail(&head, "24CS005", "Arpit Singh",  "Prof. Ravi");

    int NODE_COUNT = 5;
    Node* t[NODE_COUNT] = {NULL};

    for (int i = 0; i < NODE_COUNT; i++) {
        cudaMalloc(&t[i], sizeof(Node));
    }

    Node* t1 = head;
    Node *t2 = head->next;
    for(int i=0; i<NODE_COUNT-1;i++) {
        char *d_roll, *d_name, *d_ad;

        size_t roll_len = strlen(t1->roll) + 1;
        size_t name_len = strlen(t1->name) + 1;
        size_t ad_len   = strlen(t1->ad) + 1;

        cudaMalloc(&d_roll, roll_len);
        cudaMalloc(&d_name, name_len);
        cudaMalloc(&d_ad, ad_len);

        cudaMemcpy(d_roll, t1->roll, roll_len, cudaMemcpyHostToDevice);
        cudaMemcpy(d_name, t1->name, name_len, cudaMemcpyHostToDevice);
        cudaMemcpy(d_ad, t1->ad, ad_len, cudaMemcpyHostToDevice);

        Node temp = *t1;
        temp.roll = d_roll; // Node should contain GPU addresses. These are address copies not values
        temp.name = d_name;
        temp.ad   = d_ad;
        temp.next = t[i+1];
        cudaMemcpy(t[i], &temp, sizeof(Node), cudaMemcpyHostToDevice);
        t1=t2;
        t2=t2->next;
    }
    char *d_roll, *d_name, *d_ad;

    size_t roll_len = strlen(t1->roll) + 1;
    size_t name_len = strlen(t1->name) + 1;
    size_t ad_len   = strlen(t1->ad) + 1;

    cudaMalloc(&d_roll, roll_len);
    cudaMalloc(&d_name, name_len);
    cudaMalloc(&d_ad, ad_len);

    cudaMemcpy(d_roll, t1->roll, roll_len, cudaMemcpyHostToDevice);
    cudaMemcpy(d_name, t1->name, name_len, cudaMemcpyHostToDevice);
    cudaMemcpy(d_ad, t1->ad, ad_len, cudaMemcpyHostToDevice);

    Node temp = *t1;
    temp.roll = d_roll; // Node should contain GPU addresses. These are address copies not values
    temp.name = d_name;
    temp.ad   = d_ad;
    temp.next = NULL;
    cudaMemcpy(t[NODE_COUNT-1], &temp, sizeof(Node), cudaMemcpyHostToDevice);

    gpu_ll<<<1,1>>>(t[0]);
    cudaDeviceSynchronize();


    return 0;
}